In [66]:
import os
import sys
import numpy as np
from tqdm import tqdm
from glob import glob
import json
from collections import Counter

In [5]:



def load_json(filepath):
    with open(filepath, "r") as f:
        data = json.load(f)
        return data


# Load pre-tokenization counts
data_dir = "/scratch/shared/beegfs/piyush/datasets/text_data"
mode = "valid"
filepath = f"{data_dir}/TinyStoriesV2-GPT4-{mode}-pretokenization_counts.json"
pretok_counts = load_json(filepath)
len(pretok_counts)

13111

**Convert the strings to list of IDs using `utf-8`**

In [9]:
def _encode(string: str):
    return list(string.encode("utf-8"))


_encode("a sample string"), \
_encode("😛")

([97, 32, 115, 97, 109, 112, 108, 101, 32, 115, 116, 114, 105, 110, 103],
 [240, 159, 152, 155])

In [15]:
def pprint_list(x):
    print("\n".join(str(item) for item in x))


def show_topk_dict_entries(d, k=20):
    pprint_list(sorted(d.items(), key=lambda x: x[1], reverse=True)[:k])


show_topk_dict_entries(pretok_counts)

('.', 421616)
(',', 235432)
(' the', 211031)
(' and', 196057)
(' a', 152161)
('\n', 151989)
(' to', 150493)
(' was', 108019)
(' They', 52425)
(' it', 51670)
(' He', 49241)
(' "', 47784)
(' The', 46977)
(' said', 43900)
(' day', 43230)
(' with', 42981)
(' her', 38925)
(' his', 38766)
(' in', 38658)
(' She', 38040)


In [22]:
_encode("."), \
_encode(",")

([46], [44])

In [21]:
tok_seq_counts = {
    tuple(_encode(k)): v for k, v in pretok_counts.items()
}
show_topk_dict_entries(tok_seq_counts)

((46,), 421616)
((44,), 235432)
((32, 116, 104, 101), 211031)
((32, 97, 110, 100), 196057)
((32, 97), 152161)
((10,), 151989)
((32, 116, 111), 150493)
((32, 119, 97, 115), 108019)
((32, 84, 104, 101, 121), 52425)
((32, 105, 116), 51670)
((32, 72, 101), 49241)
((32, 34), 47784)
((32, 84, 104, 101), 46977)
((32, 115, 97, 105, 100), 43900)
((32, 100, 97, 121), 43230)
((32, 119, 105, 116, 104), 42981)
((32, 104, 101, 114), 38925)
((32, 104, 105, 115), 38766)
((32, 105, 110), 38658)
((32, 83, 104, 101), 38040)


**Count step: Count occurrences of all pairs of tokens and find max. pair**

In [36]:
def count_byte_pairs(token_sequence_counts):
    """Returns the max. frequency byte pair."""
    pair_counts = Counter()
    for k, v in token_sequence_counts.items():
        if len(k) < 2:
            # Nothing to add
            continue
        k = list(k)
        for j in range(len(k) - 1):
            pair_counts[tuple([k[j], k[j+1]])] += v
    max_pair = max(pair_counts, key=pair_counts.get)
    return max_pair

In [27]:
pair_counts = Counter()
for k, v in tok_seq_counts.items():
    if len(k) < 2:
        # Nothing to add
        continue
    k = list(k)
    for j in range(len(k) - 1):
        pair_counts[tuple([k[j], k[j+1]])] += v
max_pair = max(pair_counts, key=pair_counts.get)

len(pair_counts), max_pair

(932, (32, 116))

In [32]:
tok_seq_counts[max_pair]

2

**Merge step: Replace the max. token pair with a new token ID**

In [30]:
def _merge(k, pair, n):
    result = []
    i = 0
    while i < len(k):
        if i < len(k) - 1 and k[i] == pair[0] and k[i+1] == pair[1]:
            result.append(n)
            i += 2
        else:
            result.append(k[i])
            i += 1
    return result

def merge(ids: dict, pair: tuple, n: int):
    _ids = Counter()
    for k, v in ids.items():
        k = list(k)
        _k = _merge(k, pair, n)
        _ids[tuple(_k)] += v
    return _ids


_tok_seq_counts = merge(tok_seq_counts, max_pair, 256)
show_topk_dict_entries(_tok_seq_counts)

((46,), 421616)
((44,), 235432)
((256, 104, 101), 211031)
((32, 97, 110, 100), 196057)
((32, 97), 152161)
((10,), 151989)
((256, 111), 150493)
((32, 119, 97, 115), 108019)
((32, 84, 104, 101, 121), 52425)
((32, 105, 116), 51670)
((32, 72, 101), 49241)
((32, 34), 47784)
((32, 84, 104, 101), 46977)
((32, 115, 97, 105, 100), 43900)
((32, 100, 97, 121), 43230)
((32, 119, 105, 116, 104), 42981)
((32, 104, 101, 114), 38925)
((32, 104, 105, 115), 38766)
((32, 105, 110), 38658)
((32, 83, 104, 101), 38040)


In [35]:
_tok_seq_counts[max_pair], _tok_seq_counts[(256,)]

(0, 2)

### Training loop: naive

In [49]:
from copy import deepcopy


def count_bytes(indices: dict):
    count = 0
    for k, v in indices.items():
        count += len(k) * v
    return count


def train_bpe(
    input_path: str,
    vocab_size: int,
    special_tokens: list=['<|endoftext|>'],
):

    # Load the pre-tokenized counts
    pretok_counts = load_json(input_path)

    # Convert strings into list of bytes
    tok_seq_counts = {
        tuple(_encode(k)): v for k, v in pretok_counts.items()
    }

    # Initialize vocabulary
    vocab = {k: bytes([k]) for k in range(256)}

    # Initialize merges
    num_merges = vocab_size - 256 - len(special_tokens)
    merges = {}

    # Make a copy of the token seq. counts dict.
    indices = deepcopy(tok_seq_counts)

    # Training loop
    for i in tqdm(range(num_merges), desc="Training BPE", bar_format='{l_bar}{bar:20}{r_bar}'):

        # 1. Count the frequency of byte pairs & find max. pair
        max_pair = count_byte_pairs(indices)

        # 2. Merge the max_pair entries in the indices
        new_ind = 256 + i
        indices = merge(indices, max_pair, new_ind)
        merges[max_pair] = new_ind
        vocab[new_ind] = vocab[max_pair[0]] + vocab[max_pair[1]]

    # Add special tokens to the vocabulary
    for s in special_tokens:
        vocab[len(vocab)] = s

    # TODO: fix compute compression ratio
    # NOTE: since we have removed special tokens, those do not contribute to the n.o. bytes
    comp_ratio = count_bytes(tok_seq_counts) / count_bytes(indices)
    print(comp_ratio)

    return vocab, merges


# Test it out
vocab, merges = train_bpe(filepath, vocab_size=256+1+10000, special_tokens=['<|endoftext|>'])

Training BPE: 100%|████████████████████| 10000/10000 [04:14<00:00, 39.26it/s]

4.075876753723726


In [64]:
len(vocab), [vocab[i] for i in range(450, 455)]

(10257, [b' at', b'gether', b' cat', b' did', b' re'])

In [50]:
show_topk_dict_entries(merges)

((452, 110), 10255)
((435, 4641), 10254)
((1038, 102), 10253)
((2953, 2419), 10252)
((7633, 393), 10251)
((10249, 121), 10250)
((10248, 374), 10249)
((75, 417), 10248)
((2153, 2009), 10247)
((4260, 994), 10246)
((100, 379), 10245)
((8803, 5417), 10244)
((1082, 5486), 10243)
((536, 5190), 10242)
((5426, 39), 10241)
((307, 1463), 10240)
((119, 5848), 10239)
((1398, 115), 10238)
((7032, 7387), 10237)
((8800, 2851), 10236)


In [54]:
i = np.argmax([len(vocab[i]) for i in vocab])
print(i, vocab[i])

7311 b' accomplishment'


In [62]:
# Sanity check
import regex as re
from cs336_basics.pretokenization import find_chunk_boundaries

query = (vocab[i]).decode("utf-8")
with open("/scratch/shared/beegfs/piyush/datasets/text_data/TinyStoriesV2-GPT4-valid.txt", "rb") as f:
    num_processes = 8
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

    # The following is a serial implementation, but you can parallelize this
    # by sending each start/end pair to a set of processes.
    c = 0
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        match_iter = re.finditer(re.compile(query), chunk)
        for m in match_iter:
            c += 1
        # del chunk
    print(f"Number of times `{query}` appears in the corpus: ", c)

Number of times ` accomplishment` appears in the corpus:  14


**Optimised implementation**

In [ ]:
def _merge(k, pair, n):
    result = []
    i = 0
    while i < len(k):
        if i < len(k) - 1 and k[i] == pair[0] and k[i+1] == pair[1]:
            result.append(n)
            i += 2
        else:
            result.append(k[i])
            i += 1
    return result

def merge(ids: dict, pair: tuple, n: int):
    _ids = Counter()
    for k, v in ids.items():
        k = list(k)
        _k = _merge(k, pair, n)
        _ids[tuple(_k)] += v
    return _ids

In [93]:
def merge_optimized(indices, max_pair, new_ind, pair_counts):
    _indices = Counter()
    for k, v in indices.items():

        result = []
        i = 0
        while i < len(k):
            if i < len(k) - 1 and k[i] == max_pair[0] and k[i+1] == max_pair[1]:

                # [..... k[i - 1], **k[i], k[i+1],** k[i+2], ...]
                # 1. Replace **k[i], k[i+1]** with new_ind
                # 2. Handle left counter
                # 3. Handle right counter

                # Replacement
                result.append(new_ind)

                # Left
                if i - 1 >= 0:
                    # Decrement [k[i - 1], k[i]] by v
                    # assert (k[i - 1], k[i]) in pair_counts
                    # pair_counts[ tuple([k[i - 1], k[i]]) ] -= v
                    pair_counts[ (k[i - 1], k[i]) ] -= v

                    # Increment [k[i - 1], new_ind] by v
                    # pair_counts[ tuple([k[i - 1], new_ind]) ] += v
                    pair_counts[ (k[i - 1], new_ind) ] += v

                # Right
                if i + 2 < len(k):
                    
                    # Decrement [k[i + 1], k[i + 2]] by v
                    # assert (k[i + 1], k[i + 2]) in pair_counts
                    # pair_counts[ tuple([k[i + 1], k[i + 2]]) ] -= v
                    pair_counts[ (k[i + 1], k[i + 2]) ] -= v

                    # Increment [new_ind, k[i + 2]]
                    # pair_counts[ tuple([new_ind, k[i + 2]]) ] += v
                    pair_counts[ (new_ind, k[i + 2]) ] += v
                
                i += 2
            else:
                result.append(k[i])
                i += 1
        
        _indices[tuple(result)] += v

    return _indices, pair_counts


def train_bpe_optimized(
    input_path: str,
    vocab_size: int,
    special_tokens: list=['<|endoftext|>'],
):

    # Load the pre-tokenized counts
    pretok_counts = load_json(input_path)

    # Convert strings into list of bytes
    tok_seq_counts = {
        tuple(_encode(k)): v for k, v in pretok_counts.items()
    }

    # Initialize vocabulary
    vocab = {k: bytes([k]) for k in range(256)}

    # Initialize merges
    num_merges = vocab_size - 256 - len(special_tokens)
    merges = {}

    # Make a copy of the token seq. counts dict.
    indices = deepcopy(tok_seq_counts)

    # Maintain a single dict to count the frequency of byte pairs
    # At the start, fill it with initial counts: [a, b] = count("a, b")
    # where a/b are in [0, 255].
    pair_counts = Counter()
    for k, v in indices.items():
        if len(k) < 2:
            # Nothing to add
            continue
        for j in range(len(k) - 1):
            pair_counts[tuple([k[j], k[j+1]])] += v


    # Training loop
    for i in tqdm(range(num_merges), desc="Training BPE", bar_format='{l_bar}{bar:20}{r_bar}'):

        new_ind = 256 + i

        # Find the byte pair with max. frequency
        max_pair = max(pair_counts, key=pair_counts.get)

        # Remove max_pair from pair_counts
        del pair_counts[max_pair]

        # Update the vocab and merges
        merges[max_pair] = new_ind
        vocab[new_ind] = vocab[max_pair[0]] + vocab[max_pair[1]]

        # Merge + update the count dict
        indices, pair_counts = merge_optimized(indices, max_pair, new_ind, pair_counts)
        # import ipdb; ipdb.set_trace()
        # {k: v for k, v in indices.items() if max_pair in zip(k, k[1:])}
        # Only for debugging: checks if a particular triplet count matches
        # assert sum([v for k, v in indices.items() if (max_pair[0], max_pair[1], 111) in zip(k, k[1:], k[2:])]) == pair_counts[(new_ind, 111)]
        

    # Add special tokens to the vocabulary
    for s in special_tokens:
        vocab[len(vocab)] = s.encode('utf-8')

    # TODO: fix compute compression ratio
    # NOTE: since we have removed special tokens, those do not contribute to the n.o. bytes
    comp_ratio = count_bytes(tok_seq_counts) / count_bytes(indices)
    print(comp_ratio)

    return vocab, merges

In [94]:
# Test it out
vocab, merges = train_bpe_optimized(filepath, vocab_size=256+1+10000, special_tokens=['<|endoftext|>'])

Training BPE: 100%|████████████████████| 10000/10000 [01:57<00:00, 84.87it/s]

4.075809233654224


In [97]:
# Sanity check
i = np.argmax([len(vocab[i]) for i in vocab])
print("Vocab entry with max. length: ", i, vocab[i])
text_filepath = f"{data_dir}/TinyStoriesV2-GPT4-{mode}.txt"

import regex as re
from cs336_basics.pretokenization import find_chunk_boundaries

query = (vocab[i]).decode("utf-8")
with open(text_filepath, "rb") as f:
    num_processes = 8
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

    # The following is a serial implementation, but you can parallelize this
    # by sending each start/end pair to a set of processes.
    c = 0
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        match_iter = re.finditer(re.compile(query), chunk)
        for m in match_iter:
            c += 1
        # del chunk
print(f"Number of times `{query}` appears in the corpus: ", c)

Vocab entry with max. length:  7429 b' accomplishment'
Number of times ` accomplishment` appears in the corpus:  14


### TODOs

1. Write encoding and decoding routines (and perhaps a new class; check `minbpe` for structuring)
2. ~Optimise the code for finding max. pair~
3. Profiling code
4. Run the tests
5. Merge code for pre-tokenization and training.